# DS340 Final Project Scaffold: Combine Drill Sequences -> NFL WR Route Performance

**Question.** Does a learned representation of a WR's Combine drill movement (LSTM encoders + pooling) predict NFL route performance (e.g. separation at pass forward) *beyond* the official Combine numbers and draft position?

**Pipeline.**
```
reps in each drill family -> [LSTM encoder(s)] -> one vector per rep
        -> [pooling: mean | max | learned-weight | self-attention] -> player vector
        -> (+ optional attributes) -> [MLP head] -> one prediction per route family
```

**Feature sets compared (same downstream evaluation for all):**
- **A**: official Combine numbers + height/weight (+ draft pick as a variation)
- **B**: hand-built kinematics from the tracking data (top speed, braking, turn sharpness, ...)
- **C**: learned (LSTM / attention model)

**How the sections map to your milestones**
- Sections 1-7 (setup, audit, cleaning, tensors, targets, baselines A/B) = your **mid-project TA milestone** (aim for ~Nov 17).
- Sections 8-11 = the neural model and grouped cross-validation.
- Sections 12-13 = variations/ablations and interpretation (figures for the paper + lightning talk).

**Conventions.**
- Anything marked `VERIFY` is a column name or value I inferred from public descriptions of the data. Check it against your actual files.
- Anything marked `TODO` is yours to complete.
- **AI statement (course requirement):** this scaffold was generated with Claude. The syllabus requires you to disclose AI use and comment AI-written code. Keep the `# [AI-SCAFFOLD]` tags on lines you did not write yourselves, and delete the tag on code you rewrite. Update your written AI statement to match what you actually kept.

## 1. Setup and configuration

Keep every tunable value in `CFG` so that the ablations in section 12 are just different configs. The course runs on Colab, so install anything missing in a separate cell (for example `pip install` as a notebook command); torch, sklearn, pandas and scipy are normally preinstalled.

In [6]:
import random, math, itertools, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from scipy.stats import spearmanr

SEED = 0
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TODO: point this at the folder holding the competition CSVs.
# On Kaggle: Path("/kaggle/input/nfl-big-data-bowl-2027")
# On Colab: upload to Drive (or download with the Kaggle API) and mount it.
DATA_DIR = Path("data")

CFG = dict(
    T = 100,                  # fixed length every rep is resampled to
    channels = ["speed", "acc", "turn"],  # TODO: decide final channel list (section 5)
    min_routes_per_family = 15,   # a player needs this many routes in a family to get a target
    min_reps_per_player = 3,      # drop players with fewer usable reps
    hidden = 32,                  # LSTM hidden size (keep small: ~100 players!)
    drill_dim = 4,                # size of the drill-type embedding
    pool = "mean",                # "mean" | "max" | "weighted" | "attention"
    per_family_encoders = False,  # False = one shared LSTM; True = one LSTM per drill family
    use_attrs = True,             # feed official measurables into the head
    dropout = 0.3,
    lr = 1e-3, weight_decay = 1e-2,
    epochs = 200, patience = 20, batch_size = 16,
    n_folds = 5, n_seeds = 3,
)
print(DEVICE)

cpu


## 2. Load the data

Files you need (names from the competition data page):
- `combine_tracking.csv`: 10 Hz tracking of every Combine drill attempt (player rows and, for WR/TE drills, ball rows).
- `combine_results.csv`: official measurables and times. Gives you feature set A, `combine_position`, and (VERIFY) `draft_year`.
- `players.csv`: position, draft round/pick.
- `player_play.csv`: one row per player-play in NFL games, including `route_ran` and (VERIFY) `separation_at_pass_forward`.

**Do not load the `game_tracking_*.csv` files.** They are huge, and you do not need them: separation is already computed in `player_play.csv`. (Public write-ups also note the game tracking only contains Combine participants, not all 22 players, so you could not compute separation from defender positions anyway.)

In [9]:
import pandas as pd
players   = pd.read_csv("players.csv")
results   = pd.read_csv("combine_results.csv")
trk       = pd.read_csv("combine_tracking.csv")
play      = pd.read_csv("player_play.csv")

# TODO: print .columns and .head() for each table and fix every VERIFY below.
# Questions to answer by looking:
#  - What is the attempt/event id column called? (the code below assumes "event_id")
#  - Which column distinguishes PLAYER rows from BALL rows in combine_tracking?
#  - What is the time/frame column? (assumed "frame")
#  - Is draft year in players.csv or combine_results.csv?
for name, df in [("players", players), ("results", results), ("trk", trk), ("play", play)]:
    print(name, df.shape)
    print(df.columns)
    print(df.head())

players (510, 10)
Index(['nfl_id', 'display_name', 'draft_year', 'nfl_position', 'birth_date',
       'college_name', 'college_conference', 'draft_round',
       'draft_pick_within_round', 'draft_overall_pick'],
      dtype='object')
   nfl_id     display_name  draft_year nfl_position  birth_date  college_name  \
0   55867    Will Anderson        2023           DE  2001-09-02       Alabama   
1   55870    Paris Johnson        2023            T  2001-07-31    Ohio State   
2   55874   Darnell Wright        2023            T  2001-08-10     Tennessee   
3   55875  Peter Skoronski        2023            G  2001-07-31  Northwestern   
4   55877   Lukas Van Ness        2023           DE  2001-07-06          Iowa   

        college_conference  draft_round  draft_pick_within_round  \
0  Southeastern Conference          1.0                      3.0   
1       Big Ten Conference          1.0                      6.0   
2  Southeastern Conference          1.0                     10.0   
3      

In [10]:
wr_only_players = players[players["nfl_position"] == "WR"]
wr_only_results = results[results["nfl_id"].isin(wr_only_players["nfl_id"])]
wr_only_trk = trk[trk["nfl_id"].isin(wr_only_players["nfl_id"])]
wr_only_play = play[play["nfl_id"].isin(wr_only_players["nfl_id"])]

In [11]:
# Save WR only files, don't run every time
wr_only_players.to_csv("wr_only_players.csv", index=False)
wr_only_results.to_csv("wr_only_results.csv", index=False)
wr_only_trk.to_csv("wr_only_trk.csv", index=False)
wr_only_play.to_csv("wr_only_play.csv", index=False)

In [12]:
# Load directly from WR only csv files
wr_only_players = pd.read_csv("wr_only_players.csv")
wr_only_results = pd.read_csv("wr_only_results.csv")
wr_only_trk = pd.read_csv("wr_only_trk.csv")
wr_only_play = pd.read_csv("wr_only_play.csv")

/tmp/ipykernel_7880/2751172509.py:5: DtypeWarning: Columns (58) have mixed types. Specify dtype option on import or set low_memory=False.
  wr_only_play = pd.read_csv("wr_only_play.csv")


In [13]:
wr_only_trk['entity_type'].unique()

array(['PLAYER', 'BALL'], dtype=object)

In [14]:
wr_only_players.head()

,nfl_id,display_name,draft_year,nfl_position,birth_date,college_name,college_conference,draft_round,draft_pick_within_round,draft_overall_pick
0,55884,Jaxon Smith-Njigba,2023,WR,2002-02-14,Ohio State,Big Ten Conference,1.0,20.0,20.0
1,55885,Quentin Johnston,2023,WR,2001-09-06,Texas Christian,Big Twelve Conference,1.0,22.0,21.0
2,55887,Jordan Addison,2023,WR,2002-01-27,Southern California,Pacific Twelve Conference,1.0,24.0,23.0
3,55904,Jonathan Mingo,2023,WR,2001-04-20,Mississippi,Southeastern Conference,2.0,8.0,39.0
4,55915,Jayden Reed,2023,WR,2000-04-28,Michigan State,Big Ten Conference,2.0,19.0,50.0


In [15]:
wr_only_results.head()

,draft_year,nfl_id,combine_position,combine_height,combine_weight,hand_size,arm_length,wing_span,ten_yd_split,forty,vertical,broad_jump,three_cone,short_shuttle,bench_reps,ngs_athleticism_score,ngs_college_production_score,ngs_final_score
0,2023,56056,WR,72.250,192,9.625,31.875,76.125,1.51,4.33,NaN,NaN,NaN,NaN,NaN,76,68.0,73.0
1,2023,56118,WR,71.625,191,9.500,31.000,75.000,1.52,4.54,38.5,120.0,6.98,4.15,14.0,65,65.0,64.0
2,2023,55885,WR,74.750,208,9.625,33.625,81.625,NaN,NaN,40.5,134.0,NaN,NaN,NaN,73,79.0,81.0
3,2023,55920,WR,72.625,204,9.500,32.750,77.375,1.49,4.51,41.0,128.0,NaN,NaN,NaN,79,79.0,82.0
4,2023,56198,WR,75.125,212,9.750,34.250,79.625,1.55,4.57,35.5,123.0,NaN,4.39,NaN,70,61.0,66.0


In [16]:
wr_only_trk.head()

,draft_year,event_id,nfl_id,entity_type,time,drill_type,drill_name,attempt,x,y,s,a,dis,dir
0,2025,20250301100800_1740876338472_000,58968,PLAYER,2025-03-02T00:45:26.700,SKILL_DRILLS_WR,POST_CORNER_ROUTE_RIGHT,15,24.754100,13.583463,0.624946,3.118189,0.001323,71.210353
1,2025,20250301100800_1740876338472_000,58968,PLAYER,2025-03-02T00:45:26.800,SKILL_DRILLS_WR,POST_CORNER_ROUTE_RIGHT,15,24.825478,13.606387,0.952825,3.584155,0.005122,70.563882
2,2025,20250301100800_1740876338472_000,58968,PLAYER,2025-03-02T00:45:26.900,SKILL_DRILLS_WR,POST_CORNER_ROUTE_RIGHT,15,24.937019,13.638515,1.339290,3.997314,0.012484,70.829717
3,2025,20250301100800_1740876338472_000,58968,PLAYER,2025-03-02T00:45:27.000,SKILL_DRILLS_WR,POST_CORNER_ROUTE_RIGHT,15,25.081548,13.689702,1.752842,4.333521,0.021052,70.606116
4,2025,20250301100800_1740876338472_000,58968,PLAYER,2025-03-02T00:45:27.100,SKILL_DRILLS_WR,POST_CORNER_ROUTE_RIGHT,15,25.270650,13.764353,2.215085,4.603238,0.036191,70.224139


In [17]:
acceleration_values = wr_only_trk['a'].unique()
print(sorted(acceleration_values)[:5])
# Acceleration is only done in positive terms

[np.float64(0.0), np.float64(0.003333333), np.float64(0.004714045), np.float64(0.007115501), np.float64(0.00745356)]


In [18]:
wr_skill_drills = wr_only_trk[wr_only_trk['drill_type'] == 'SKILL_DRILLS_WR']
wr_skill_drills['drill_name'].unique()
# We have a SPEED_OUT_ROUTE_LEFT and SPEED_OUT_LEFT
# Probably best to leave out GAUNTLET_DRILL and OVER_SHOULDER_ADJUST, since they aren't real routes

array(['POST_CORNER_ROUTE_RIGHT', 'GAUNTLET_DRILL',
       'OVER_SHOULDER_ADJUST', 'GO_ROUTE_RIGHT', 'CIRCUS_ROUTE_LEFT',
       'SLANT_ROUTE_LEFT', 'SPEED_OUT_ROUTE_LEFT', 'CURL_ROUTE_RIGHT',
       'DAGGER_ROUTE_LEFT', 'SLOT_STRIKE_ROUTE_LEFT',
       'COMEBACK_ROUTE_RIGHT', 'SLOT_SAIL_ROUTE_RIGHT',
       'RED_ZONE_FADE_RIGHT', 'SPEED_OUT_LEFT', 'POST_CORNER_ROUTE_LEFT'],
      dtype=object)

In [19]:
def parse_drill(name):
    n = name.replace("_ROUTE", "")             # merges SPEED_OUT_ROUTE_LEFT / SPEED_OUT_LEFT
    side = "LEFT" if n.endswith("_LEFT") else "RIGHT" if n.endswith("_RIGHT") else None
    base = n.rsplit("_", 1)[0] if side else n
    return base, side

# e.g. "POST_CORNER_ROUTE_LEFT" -> ("POST_CORNER", "LEFT"); "GAUNTLET_DRILL" -> ("GAUNTLET_DRILL", None)

In [20]:
wr_only_play.head()

,game_id,play_id,team_abbr,nfl_id,lined_up_position,play_description,quarter,down,yards_to_go,possession_team,...,sack,tackle,assist,caused_forced_fumble,recovered_fumble,tackle_for_loss,time_to_pressure,time_to_qb_hurry,coverage_assignment,coverage_assignment_at_snap
0,2025112400,3360,CAR,57606,SLOT_WR,(4:08) (Shotgun) B.Young pass incomplete deep ...,4,1,10,CAR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025092112,2636,SF,57152,SLOT_WR,(6:32) M.Jones pass deep middle to R.Pearsall ...,3,1,10,SF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025113008,2609,LV,55965,WR,(2:18) G.Smith pass short right to A.Jeanty pu...,3,1,10,LV,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024090807,1715,MIN,55887,SLOT_WR,(1:21) S.Darnold pass short right to J.Jeffers...,2,1,10,MIN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025090702,3505,MIA,57304,WR,(1:56) (Shotgun) Z.Wilson pass short middle to...,4,3,4,MIA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# Keep rows where the player actually ran a route, then flag whether separation exists
routes = wr_only_play[wr_only_play["route_ran"].notna()].copy()
routes["has_sep"] = routes['separation_at_pass_forward'].notna()

print("WR route rows:", len(routes))
print("Overall non-null rate:", round(routes["has_sep"].mean(), 3))

# By route type: how often is separation recorded?
by_route = (routes.groupby("route_ran")["has_sep"]
            .agg(n="size", non_null="sum", rate="mean")
            .sort_values("n", ascending=False))
print(by_route)

# By player: how many usable values does each WR have?
by_player = routes.groupby("nfl_id")["has_sep"].agg(n_routes="size", n_sep="sum", rate="mean")
print(by_player["rate"].describe())
print("WRs with >= 15 separation values:", int((by_player["n_sep"] >= 15).sum()))

WR route rows: 42806
Overall non-null rate: 0.877
              n  non_null      rate
route_ran                          
GO         8533      7587  0.889136
HITCH      6980      6128  0.877937
CROSS      5374      4569  0.850205
IN         4829      4103  0.849658
POST       4310      3699  0.858237
OUT        4141      3629  0.876358
CORNER     3427      2930  0.854975
SLANT      2850      2645  0.928070
SCREEN     1169      1134  0.970060
FLAT       1110      1023  0.921622
WHEEL        59        55  0.932203
ANGLE        24        23  0.958333
count    105.000000
mean       0.873416
std        0.050189
min        0.675676
25%        0.848485
50%        0.882690
75%        0.900763
max        1.000000
Name: rate, dtype: float64
WRs with >= 15 separation values: 102


In [22]:
# Possible route families, group to improve consistency between different players (some may rarely run some routes)
ROUTE_FAMILY = {
    "GO": "vertical", "POST": "vertical", "CORNER": "vertical", "WHEEL": "vertical",
    "OUT": "break", "IN": "break", "CROSS": "break", "SLANT": "break", "ANGLE": "break",
    "HITCH": "short",
    # "FLAT" and "SCREEN" left out on purpose
}
routes["fam"] = routes["route_ran"].map(ROUTE_FAMILY)

have = routes[routes["has_sep"] & routes["fam"].notna()]
per = have.groupby(["nfl_id", "fam"]).size().unstack(fill_value=0)
for thr in [10, 15, 25]:
    print(f"WRs with >= {thr} values per family:")
    print((per >= thr).sum().to_string(), "\n")

WRs with >= 10 values per family:
fam
break       95
short       78
vertical    95 

WRs with >= 15 values per family:
fam
break       86
short       70
vertical    86 

WRs with >= 25 values per family:
fam
break       76
short       63
vertical    79 



## 3. Data audit (do this before designing anything else)

This decides your drills, your encoder layout and your split. Answer these with tables:

1. For each drill (`drill_type`, `drill_name`), how many **WRs** and how many **reps** are there, broken down by **draft year**? A drill run in only one year is confounded with draft class.
2. Distribution of **reps per WR** (overall and per drill family). The ~14 reps/WR figure is an average, and some families may have 1-2 reps per player.
3. Which WR drills have **ball frames**? (Useful for alignment events such as the catch.)
4. For each WR, how many **NFL routes with separation recorded**, per route family? How many WRs clear `min_routes_per_family`?
5. How many WRs end up with **both** usable Combine reps and usable targets?

**Decision gates (suggested; adjust with the TAs):**
- A drill family with fewer than ~100-200 total reps cannot train its own LSTM. Use the shared encoder, or fall back to hand-built features for that family.
- If fewer than ~60 WRs have usable targets in a route family, drop that family from the multi-output head.
- If a draft-class holdout leaves fewer than ~20 test WRs, use K-fold over players instead and report the holdout as a secondary check.

In [23]:
# [AI-SCAFFOLD] Build one row per drill attempt, attached to player info.
KEYS = ["event_id", "nfl_id"]          # VERIFY attempt-id column name

# TODO: if tracking has both PLAYER and BALL rows, split them here (VERIFY the column).
trk_player = wr_only_trk[wr_only_trk["entity_type"] == "PLAYER"]
trk_ball   = wr_only_trk[wr_only_trk["entity_type"] == "BALL"]

trk_player["time"] = pd.to_datetime(trk_player["time"])

wr_attempts = (trk_player.groupby(KEYS)
               .agg(drill_type=("drill_type", "first"), drill_name=("drill_name", "first"),
                    draft_year=("draft_year", "first"), attempt=("attempt", "first"),
                    n_frames=("time", "count"))
               .reset_index())

# TODO 3.1: players per (drill_type, drill_name) x draft_year, and reps per (drill) x draft_year
print("Player Counts")
print(wr_attempts.pivot_table(index=["drill_type", "drill_name"], columns="draft_year",
                              values="nfl_id", aggfunc="nunique", fill_value=0))
print()
print("Rep Counts")
print(wr_attempts.pivot_table(index=["drill_type", "drill_name"], columns="draft_year",
                              values="event_id", aggfunc="nunique", fill_value=0))
# Can't use 3 cone or shuttle due to low adoption rates :(

Player Counts
draft_year                                2023  2024  2025
drill_type       drill_name                               
FORTY_YARD_DASH  FORTY_YARD_DASH            32    28    33
SHORT_SHUTTLE    SHORT_SHUTTLE              12     8     9
SKILL_DRILLS_WR  CIRCUS_ROUTE_LEFT           0     0    37
                 COMEBACK_ROUTE_RIGHT       35    28    34
                 CURL_ROUTE_RIGHT           36    30    36
                 DAGGER_ROUTE_LEFT          38    31    37
                 GAUNTLET_DRILL             38    31    37
                 GO_ROUTE_RIGHT             34    29    34
                 OVER_SHOULDER_ADJUST       36    29    34
                 POST_CORNER_ROUTE_LEFT     38    31     0
                 POST_CORNER_ROUTE_RIGHT    35    29    31
                 RED_ZONE_FADE_RIGHT        35    29    33
                 SLANT_ROUTE_LEFT           38    31    37
                 SLOT_SAIL_ROUTE_RIGHT      32    30    35
                 SLOT_STRIKE_ROUTE_LEFT   

/tmp/ipykernel_7880/3615082982.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trk_player["time"] = pd.to_datetime(trk_player["time"])


In [24]:
# TODO 3.2: reps per WR, overall and per drill family (describe() + histogram)
reps_per_wr = wr_attempts.groupby(["nfl_id", "drill_type"]).size().unstack(fill_value=0)
print(reps_per_wr.describe())
# TODO 3.3: which WR drills have ball frames?
print()
ball = wr_only_trk[wr_only_trk["entity_type"] != "PLAYER"]
print(ball.groupby("drill_name")["event_id"].nunique())

drill_type  FORTY_YARD_DASH  SHORT_SHUTTLE  SKILL_DRILLS_WR  THREE_CONE_DRILL
count            106.000000     106.000000       106.000000        106.000000
mean               1.641509       0.339623        14.273585          0.283019
std                0.692665       0.615464         1.889989          0.581537
min                0.000000       0.000000         8.000000          0.000000
25%                2.000000       0.000000        14.000000          0.000000
50%                2.000000       0.000000        14.000000          0.000000
75%                2.000000       1.000000        15.000000          0.000000
max                2.000000       3.000000        18.000000          3.000000

drill_name
CIRCUS_ROUTE_LEFT          37
COMEBACK_ROUTE_RIGHT       36
CURL_ROUTE_RIGHT           33
DAGGER_ROUTE_LEFT          37
GO_ROUTE_RIGHT             30
OVER_SHOULDER_ADJUST       32
POST_CORNER_ROUTE_RIGHT    37
RED_ZONE_FADE_RIGHT        29
SLANT_ROUTE_LEFT           36
SLOT_SAIL_ROUTE_

In [28]:
MIN_SEP_VALUES = 15           # separation values needed per family to count as a target
MIN_ROUTE_REPS = 3     # route-drill reps needed per , consider dropping based on performance to include more players
EXCLUDE_BASE = {"CIRCUS", "GAUNTLET_DRILL", "OVER_SHOULDER_ADJUST"}   # not in the v1 encoder

# ---- 3.4: separation values per WR per route family ----
per = (have.groupby(["nfl_id", "fam"]).size().unstack(fill_value=0)
       .reindex(sorted(wr_ids), fill_value=0))          # include WRs with no separation data at all
print("WRs meeting MIN_SEP, by family:")
print((per >= MIN_SEP).sum())
print("\nNumber of WRs by count of qualifying families:")
print((per >= MIN_SEP).sum(axis=1).value_counts().sort_index())

# ---- 3.5: intersection of Combine reps and separation targets ----
wa = wr_attempts.copy()
wa["base"] = wa["drill_name"].map(lambda n: parse_drill(n)[0])

route_reps = (wa[(wa["drill_type"] == "SKILL_DRILLS_WR") & (~wa["base"].isin(EXCLUDE_BASE))]
              .groupby("nfl_id").size().rename("route_reps"))
forty_reps = (wa[wa["drill_type"] == "FORTY_YARD_DASH"]
              .groupby("nfl_id").size().rename("forty_reps"))

tab = (pd.DataFrame(index=sorted(wr_ids))
       .join(results.set_index("nfl_id")[["draft_year"]])
       .join(route_reps).join(forty_reps).join(per))
tab[["route_reps", "forty_reps"]] = tab[["route_reps", "forty_reps"]].fillna(0)

fams = list(per.columns)
tab[fams] = tab[fams].fillna(0)
tab["has_reps"] = tab["route_reps"] >= MIN_ROUTE_REPS
tab["n_fams"] = (tab[fams] >= MIN_SEP).sum(axis=1)
usable = tab[tab["has_reps"] & (tab["n_fams"] >= 1)]

print("WRs in combine_results:        ", len(tab))
print("with enough route reps:        ", int(tab["has_reps"].sum()))
print("with >= 1 family target:       ", int((tab["n_fams"] >= 1).sum()))
print("usable (both):                 ", len(usable))
print("\nusable WRs per family (reps AND target):")
print({f: int(((tab[f] >= MIN_SEP) & tab["has_reps"]).sum()) for f in fams})
print("\nusable WRs by draft class:")
print(usable["draft_year"].value_counts().sort_index())

# Who is dropped, and why
print(tab[~tab.index.isin(usable.index)][["draft_year", "route_reps", "forty_reps"] + fams])

NameError: name 'wr_ids' is not defined

In [ ]:
wr_res = results[results["nfl_id"].isin(wr_ids)]
print(wr_res.columns)
print(len(wr_res))
print(wr_res[["forty", "three_cone", "short_shuttle", "vertical", "broad_jump"]]
      .notna().sum())

Index(['draft_year', 'nfl_id', 'combine_position', 'combine_height',
       'combine_weight', 'hand_size', 'arm_length', 'wing_span',
       'ten_yd_split', 'forty', 'vertical', 'broad_jump', 'three_cone',
       'short_shuttle', 'bench_reps', 'ngs_athleticism_score',
       'ngs_college_production_score', 'ngs_final_score'],
      dtype='object')
107
forty            97
three_cone       33
short_shuttle    38
vertical         96
broad_jump       95
dtype: int64


## 4. Cleaning and kinematics

Known quirks from public descriptions of the data (confirm each one yourself):
- The provided **distance column under-counts**: recompute distance from `x`, `y`.
- **Acceleration is unsigned**: compute signed acceleration from the speed column (decelerations must be negative for braking features).
- **Drill names drift across years** (e.g. `LINE` vs `LINE_DRILL`): build an alias map and canonicalize.
- Some attempts are **glitched or truncated**: flag them (impossible speeds, too few frames, ends mid-drill) and drop or log them.
- Combine clips **start in motion**, so raw-clock times differ from official times. Never compare raw durations to laser times; align on events (section 5).

Turn rate from heading is noisy when the player is nearly stationary. Gate it by speed (for example set it to 0 below ~1 yd/s) or smooth it.

In [ ]:
import numpy as np

KEYS = ["event_id", "nfl_id"]
d = trk_player.sort_values(KEYS + ["time"]).copy()
g = d.groupby(KEYS)
d["dt"] = g["time"].diff().dt.total_seconds()
d["acc_fd"] = g["s"].diff() / d["dt"]
ok = d["acc_fd"].notna()

print("a has negatives:", (d["a"] < 0).any())                       # False means unsigned
print("corr(|finite diff|, a):", np.corrcoef(d.loc[ok, "acc_fd"].abs(), d.loc[ok, "a"])[0, 1])

d["acc_signed"] = np.sign(d["acc_fd"]).fillna(0) * d["a"]           # sign from speed change, size from sensor
rad = np.radians(d["dir"])
d["dir_sin"], d["dir_cos"] = np.sin(rad), np.cos(rad)
d["turn"] = ((g["dir"].diff() + 180) % 360 - 180) / d["dt"]         # deg/s, wrap-safe
d.loc[d["s"] < 1.0, "turn"] = 0.0                                   # gate noisy heading at low speed

a has negatives: False
corr(|finite diff|, a): 0.5715470568082661


In [ ]:
g = d.groupby(KEYS)
d["s_sm"] = g["s"].transform(lambda x: x.rolling(3, center=True, min_periods=1).mean())  # light smoothing
g = d.groupby(KEYS)
d["acc_tan"] = g["s_sm"].diff() / d["dt"]              # signed, along the path
omega = np.radians(d["turn"])                          # rad/s (turn was deg/s)
d["acc_cent"] = d["s"] * omega                         # sideways (centripetal) part
d["acc_total"] = np.sqrt(d["acc_tan"] ** 2 + d["acc_cent"] ** 2)

m = d["acc_tan"].notna() & (d["s"] >= 1.0)
for name in ["acc_tan", "acc_total"]:
    print(name, np.corrcoef(d.loc[m, name].abs(), d.loc[m, "a"])[0, 1])

acc_tan 0.5527810510972182
acc_total 0.9832819073463597


In [ ]:
# TODO: fill after inspecting drill_name values in the audit. Map every variant to one canonical name.

ROUTE_FAMILY = {
    "GO": "vertical", "POST": "vertical", "CORNER": "vertical", "WHEEL": "vertical",
    "OUT": "break", "IN": "break", "CROSS": "break", "SLANT": "break", "ANGLE": "break",
    "HITCH": "short",
    # "FLAT" and "SCREEN" left out on purpose
}

# Drill families (VERIFY names against drill_type values on the data page).
# Families decide which reps share an LSTM when per_family_encoders=True.
FAMILY_OF_DRILL_TYPE = {
    "FORTY_YARD_DASH": "sprint",
    "SKILL_DRILLS_WR": "route",
}
FAMILIES = sorted(set(FAMILY_OF_DRILL_TYPE.values()))
FAMILY_ID = {f: i for i, f in enumerate(FAMILIES)}

def add_kinematics(df):
    """Per-attempt channels: speed, acc_tan (signed, along path), acc_cent (signed, sideways)."""
    d = df.sort_values(KEYS + ["time"]).copy()
    g = d.groupby(KEYS)
    d["dt"] = g["time"].diff().dt.total_seconds().fillna(0.1).clip(lower=1e-3)
    d["speed"] = d["s"]                                  # find_segment() expects a "speed" column
    d["s_sm"] = g["s"].transform(lambda x: x.rolling(3, center=True, min_periods=1).mean())
    g = d.groupby(KEYS)
    d["acc_tan"] = (g["s_sm"].diff() / d["dt"]).fillna(0.0)
    turn = ((g["dir"].diff() + 180) % 360 - 180) / d["dt"]          # deg/s, wrap-safe
    d["acc_cent"] = (d["s"] * np.radians(turn)).fillna(0.0)         # signed sideways acceleration
    d.loc[d["s"] < 1.0, "acc_cent"] = 0.0                           # heading is unreliable at low speed
    return d

CFG["channels"] = ["speed", "acc_tan", "acc_cent"]

def flag_bad_attempts(df):
    """TODO: return a set of (event_id, nfl_id) to drop. Ideas: peak speed above a plausible max,
    fewer than N frames, big gaps in frames, attempt ends before the drill is complete."""
    return set()

wr_trk = trk_player.merge(attempts[KEYS + ["combine_position", "draft_year"]], on=KEYS)
wr_trk = wr_trk[wr_trk["combine_position"] == "WR"]
wr_trk["drill_name"] = wr_trk["drill_name"].replace(DRILL_ALIASES)
wr_trk = add_kinematics(wr_trk)
bad = flag_bad_attempts(wr_trk)
wr_trk = wr_trk[~wr_trk.set_index(KEYS).index.isin(bad)]

## 5. Turn each rep into a fixed-size tensor

Each rep becomes an array of shape `(T, C)`:
1. **Trim to the movement**: start at the first frame where speed exceeds a small threshold (the clip may start in motion, so also handle the case where it already is).
2. **Pick alignment events per family**, then resample the segment(s) between events to fixed length `T`:
   - `sprint`: motion start -> end of the run. (Entrants report the first ~10 yd of the 40 are unreliable in tracking; consider ignoring official-time comparisons.)
   - `cod` (3-cone, shuttle): motion start -> end. Optionally align on each direction change.
   - `route`: motion start -> break point (largest heading change) -> catch/end. If ball frames exist, use the ball to find the catch.
3. **Channels**: speed, signed acceleration, turn rate (add more if useful, e.g. position relative to the start).
4. **Duration** is lost by resampling, so append it as one extra *constant* channel.
5. **Standardize channels using training-fold statistics only** (done in section 9, not here; fitting scalers on all reps leaks test information).

Output: a list of rep dicts that every later section consumes.

In [ ]:
def resample(arr, T):
    """[AI-SCAFFOLD] Linearly resample a (n, C) array to (T, C)."""
    n = len(arr)
    if n == 1:
        return np.repeat(arr, T, axis=0)
    src = np.linspace(0.0, 1.0, n)
    dst = np.linspace(0.0, 1.0, T)
    return np.stack([np.interp(dst, src, arr[:, c]) for c in range(arr.shape[1])], axis=1)

def find_segment(rep_df, family):
    """TODO: return (start_idx, end_idx) of the part of the rep to keep, using family-specific events.
    Minimal version: first frame with speed > 0.5 through the last frame with speed > 0.5."""
    moving = np.where(rep_df["speed"].to_numpy() > 0.5)[0]
    if len(moving) < 5:
        return None
    return moving[0], moving[-1] + 1

# Build a stable integer id for every canonical drill name (used for the drill embedding).
DRILL_IDS = {d: i for i, d in enumerate(sorted(wr_trk["drill_name"].unique()))}

def build_reps(df, T=CFG["T"], channels=CFG["channels"]):
    """[AI-SCAFFOLD] One dict per usable rep: x (T, C+1), drill_id, family_id, nfl_id, draft_year."""
    reps = []
    for (ev, pid), rep in df.groupby(KEYS):
        dt_type = rep["drill_type"].iloc[0]
        family = FAMILY_OF_DRILL_TYPE.get(dt_type)
        if family is None:
            continue                                  # drill types we are not using
        seg = find_segment(rep, family)
        if seg is None:
            continue
        a, b = seg
        arr = rep[channels].to_numpy()[a:b]
        duration = (b - a) * 0.1
        x = resample(arr, T)
        x = np.concatenate([x, np.full((T, 1), duration)], axis=1)   # constant duration channel
        reps.append(dict(x=x.astype(np.float32), nfl_id=pid, draft_year=rep["draft_year"].iloc[0],
                         drill_id=DRILL_IDS[rep["drill_name"].iloc[0]], family_id=FAMILY_ID[family]))
    return reps

reps = build_reps(wr_trk)
print(len(reps), "reps for", len({r["nfl_id"] for r in reps}), "WRs")
# TODO: plot ~6 reps per family (speed vs resampled time) and sanity check the alignment visually.

## 6. Targets from `player_play.csv`

Goal: one number per (player, route family): average **separation at pass forward** on routes of that family.

Design decisions to make and write down:
- **Route families.** Group `route_ran` values into a few families so each has enough routes per player. The page lists GO, OUT, SLANT, CROSS, POST, CORNER, HITCH, SCREEN, FLAT, WHEEL; there may be others, so print the unmapped ones.
- **Noise control.** A player's average over 10 routes is very noisy. Use a minimum route count (`min_routes_per_family`), weight the loss by route count, and optionally **shrink** each player's mean toward the family mean (empirical Bayes).
- **Context adjustment (optional but valuable).** Separation depends on coverage (man/zone) and cushion at the snap. Fit a simple regression of separation on route type + coverage + cushion and use the *residual*; otherwise a WR who mostly faces zone looks "better".
- **Mask.** If a player has no qualifying routes in a family, that target is missing: store `ym = 0` so the loss ignores it.
- Only use **regular-season** rows if the table includes preseason ids (VERIFY how to filter).

In [ ]:
# TODO: fill after printing play["route_ran"].value_counts(). Every route must map to one family.
ROUTE_FAMILY = {
    # "GO": "vertical", "POST": "vertical", "CORNER": "vertical", "WHEEL": "vertical",
    # "OUT": "break", "SLANT": "break", "CROSS": "break",
    # "HITCH": "short", "FLAT": "short", "SCREEN": "short",
}
ROUTE_FAMS = sorted(set(ROUTE_FAMILY.values()))
SEP_COL = "separation_at_pass_forward"     # VERIFY

def build_targets(play_df, wr_ids):
    """[AI-SCAFFOLD] Returns dict nfl_id -> (y, ym, w), each of length K = len(ROUTE_FAMS)."""
    p = play_df[play_df["nfl_id"].isin(wr_ids)].dropna(subset=["route_ran", SEP_COL]).copy()
    p["fam"] = p["route_ran"].map(ROUTE_FAMILY)
    unmapped = p.loc[p["fam"].isna(), "route_ran"].unique()
    if len(unmapped):
        print("Unmapped routes (add them to ROUTE_FAMILY):", unmapped)
    p = p.dropna(subset=["fam"])
    # TODO (optional): replace SEP_COL by a residual after adjusting for coverage and cushion.
    out = {}
    for pid, g in p.groupby("nfl_id"):
        y = np.zeros(len(ROUTE_FAMS), dtype=np.float32)
        ym = np.zeros(len(ROUTE_FAMS), dtype=np.float32)
        w = np.zeros(len(ROUTE_FAMS), dtype=np.float32)
        for k, fam in enumerate(ROUTE_FAMS):
            gg = g[g["fam"] == fam]
            if len(gg) >= CFG["min_routes_per_family"]:
                y[k], ym[k], w[k] = gg[SEP_COL].mean(), 1.0, math.sqrt(len(gg))   # TODO: shrinkage
        if ym.sum() > 0:
            out[pid] = (y, ym, w)
    return out

wr_ids = set(results.loc[results["combine_position"] == "WR", "nfl_id"])
targets = build_targets(play, wr_ids)
print(len(targets), "WRs with at least one qualifying route family")

## 7. Baselines A and B with the evaluation harness (your milestone)

Everything here is scored with the **same folds and metrics** later used for the neural model, so numbers are comparable.

**Splits.** Players are the unit. Default: shuffled K-fold over players (each player appears in exactly one test fold). Secondary check: **leave one draft class out** (the realistic "predict the next class" test), if the audit shows enough WRs per class.

**Metrics.** Per route family: Spearman correlation between predicted and actual (ranking is what a scout cares about) and R^2. Report the mean across families and the per-family values, with a bootstrap over players for confidence intervals (n is small, so intervals matter).

**Baselines.**
- **A**: official numbers (40, 3-cone, shuttle, vertical, broad jump, height, weight; draft pick as a variation). Ridge per route family.
- **B**: hand-built kinematics per player: top speed, time to 90% of top speed, peak acceleration, hardest deceleration, peak turn rate, speed retained after the break. Ridge (or a small random forest from class) per route family.

If A or B matches the neural model, that is a result worth reporting.

In [ ]:
def make_player_table(reps, targets, results):
    """[AI-SCAFFOLD] One entry per WR that has targets and >= min_reps_per_player reps."""
    by_player = {}
    for r in reps:
        by_player.setdefault(r["nfl_id"], []).append(r)
    ids = [pid for pid in targets
           if len(by_player.get(pid, [])) >= CFG["min_reps_per_player"]]
    return ids, by_player

player_ids, reps_by_player = make_player_table(reps, targets, results)
Y  = np.stack([targets[p][0] for p in player_ids])
YM = np.stack([targets[p][1] for p in player_ids])
W  = np.stack([targets[p][2] for p in player_ids])
draft_year = results.set_index("nfl_id").loc[player_ids, "draft_year"].to_numpy()   # VERIFY

def get_splits(n, mode="kfold", seed=SEED):
    """Yield (train_idx, test_idx). mode: 'kfold' or 'draft_class'."""
    if mode == "kfold":
        yield from KFold(CFG["n_folds"], shuffle=True, random_state=seed).split(np.arange(n))
    else:
        for yr in np.unique(draft_year):
            test = np.where(draft_year == yr)[0]
            train = np.where(draft_year != yr)[0]
            yield train, test

def score(pred, Y, YM, W=None):
    """Spearman and R^2 per route family, using only players with a target for that family."""
    rows = []
    for k, fam in enumerate(ROUTE_FAMS):
        m = YM[:, k] > 0
        if m.sum() < 5:
            continue
        rho = spearmanr(pred[m, k], Y[m, k]).correlation
        ss_res = ((Y[m, k] - pred[m, k]) ** 2).sum()
        ss_tot = ((Y[m, k] - Y[m, k].mean()) ** 2).sum()
        rows.append(dict(family=fam, n=int(m.sum()), spearman=rho, r2=1 - ss_res / ss_tot))
    return pd.DataFrame(rows)

def ridge_oof(X, Y, YM, mode="kfold", alpha=10.0):
    """Out-of-fold predictions from per-family ridge. X: (n_players, n_features)."""
    oof = np.full_like(Y, np.nan)
    for tr, te in get_splits(len(X), mode):
        mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-8        # scale with TRAIN stats only
        for k in range(Y.shape[1]):
            trk_ = tr[YM[tr, k] > 0]
            if len(trk_) < 10:
                continue
            m = Ridge(alpha=alpha).fit((X[trk_] - mu) / sd, Y[trk_, k])
            oof[te, k] = m.predict((X[te] - mu) / sd)
    return oof

# --- Feature set A ---------------------------------------------------------------------------
# TODO: choose official columns from combine_results (VERIFY names) and build X_A, shape (n_players, n_features).
# Handle opt-outs / missing values (e.g. impute with TRAIN medians inside each fold, plus a missing flag).
# X_A = results.set_index("nfl_id").loc[player_ids, [...]].to_numpy()

# --- Feature set B ---------------------------------------------------------------------------
# TODO: per rep, compute the kinematic summaries, then take the per-player mean (or best rep).
# X_B = ...

# --- Run and record ---------------------------------------------------------------------------
# resA = score(ridge_oof(X_A, Y, YM), Y, YM); print(resA)
# resB = score(ridge_oof(X_B, Y, YM), Y, YM); print(resB)
# TODO: also run A+draft pick, and the "draft pick only" baseline.

## 8. Dataset and batching (variable number of reps per player)

A training example is a **player**: a set of reps (each `(T, C)`), their drill and family ids, an attribute vector, and the targets with their mask and weights.

Players have different numbers of reps, so the collate function pads to the largest count in the batch and returns a boolean `mask` marking real reps. Every later module must respect that mask (padding must never influence pooling or attention).

Standardization of channels and attributes uses statistics computed from the **training fold only** and is passed into the dataset.

In [ ]:
def fit_scalers(train_player_ids, reps_by_player, attrs):
    """[AI-SCAFFOLD] Channel and attribute mean/std from TRAIN players only."""
    allx = np.concatenate([r["x"] for p in train_player_ids for r in reps_by_player[p]], axis=0)  # (N*T, C)
    ch_mean, ch_std = allx.mean(0), allx.std(0) + 1e-6
    return ch_mean, ch_std, attrs.mean(0), attrs.std(0) + 1e-6

class PlayerDataset(Dataset):
    def __init__(self, idx, player_ids, reps_by_player, attrs, Y, YM, W, scalers):
        self.idx, self.player_ids = idx, player_ids
        self.reps_by_player, self.attrs = reps_by_player, attrs
        self.Y, self.YM, self.W = Y, YM, W
        self.ch_mean, self.ch_std, self.a_mean, self.a_std = scalers
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        reps = self.reps_by_player[self.player_ids[j]]
        x = np.stack([(r["x"] - self.ch_mean) / self.ch_std for r in reps])      # (R, T, C)
        return dict(
            x=torch.tensor(x, dtype=torch.float32),
            drill=torch.tensor([r["drill_id"] for r in reps]),
            fam=torch.tensor([r["family_id"] for r in reps]),
            attrs=torch.tensor((self.attrs[j] - self.a_mean) / self.a_std, dtype=torch.float32),
            y=torch.tensor(self.Y[j]), ym=torch.tensor(self.YM[j]), w=torch.tensor(self.W[j]))

def collate(batch):
    """Pad the rep dimension to the max count in the batch and build the validity mask."""
    B = len(batch)
    R = max(b["x"].shape[0] for b in batch)
    T, C = batch[0]["x"].shape[1:]
    x = torch.zeros(B, R, T, C)
    drill = torch.zeros(B, R, dtype=torch.long)
    fam = torch.zeros(B, R, dtype=torch.long)
    mask = torch.zeros(B, R, dtype=torch.bool)
    for i, b in enumerate(batch):
        r = b["x"].shape[0]
        x[i, :r], drill[i, :r], fam[i, :r], mask[i, :r] = b["x"], b["drill"], b["fam"], True
    out = dict(x=x, drill=drill, fam=fam, mask=mask)
    for k in ["attrs", "y", "ym", "w"]:
        out[k] = torch.stack([b[k] for b in batch])
    return out

## 9. Model

Three parts, each a variation point:

1. **RepEncoder**: an LSTM over one rep. The drill embedding is concatenated to *every timestep*, so a shared LSTM can still behave differently per drill. With `per_family_encoders=True` each drill family gets its own LSTM (bias-variance tradeoff: more specialization, less data per model).
2. **Pooling** over the reps of a player, respecting the mask: `mean`, `max`, `weighted` (one learned score per rep, softmax) or `attention` (a single small self-attention block, then mean over valid reps).
3. **Head**: an MLP that outputs one value per route family (multi-output), with optional attributes concatenated to the player vector.

Gotchas:
- A player with **zero** valid reps would give NaNs in attention (all keys masked). Section 7 already drops players below `min_reps_per_player`.
- Keep the model tiny. About 100 players cannot support a large network.

In [ ]:
class RepEncoder(nn.Module):
    """[AI-SCAFFOLD] LSTM over one rep; drill embedding appended at every timestep."""
    def __init__(self, n_channels, n_drills, drill_dim, hidden, dropout):
        super().__init__()
        self.drill_emb = nn.Embedding(n_drills, drill_dim)
        self.lstm = nn.LSTM(n_channels + drill_dim, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, drill_id):                      # x: (N, T, C), drill_id: (N,)
        d = self.drill_emb(drill_id).unsqueeze(1).expand(-1, x.size(1), -1)
        _, (h, _) = self.lstm(torch.cat([x, d], dim=-1))
        return self.drop(h[-1])                          # (N, hidden)

class Pool(nn.Module):
    """[AI-SCAFFOLD] Mask-aware pooling over reps. h: (B, R, D), mask: (B, R) True = real rep."""
    def __init__(self, kind, dim, heads=2, dropout=0.1):
        super().__init__()
        self.kind = kind
        self.last_weights = None                         # saved for interpretation (section 13)
        if kind == "weighted":
            self.score = nn.Linear(dim, 1)
        if kind == "attention":
            self.mha = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
            self.norm = nn.LayerNorm(dim)
    def forward(self, h, mask):
        m = mask.unsqueeze(-1).float()
        if self.kind == "mean":
            return (h * m).sum(1) / m.sum(1).clamp(min=1)
        if self.kind == "max":
            return h.masked_fill(~mask.unsqueeze(-1), -1e9).max(1).values
        if self.kind == "weighted":
            s = self.score(h).squeeze(-1).masked_fill(~mask, -1e9)
            w = torch.softmax(s, dim=1)
            self.last_weights = w.detach()
            return (w.unsqueeze(-1) * h).sum(1)
        if self.kind == "attention":
            out, w = self.mha(h, h, h, key_padding_mask=~mask, need_weights=True)
            self.last_weights = w.detach()               # (B, R, R); average over queries to rank reps
            h2 = self.norm(h + out)                      # residual + norm
            return (h2 * m).sum(1) / m.sum(1).clamp(min=1)
        raise ValueError(self.kind)

class PlayerModel(nn.Module):
    def __init__(self, n_channels, n_drills, n_attrs, n_out, cfg):
        super().__init__()
        self.cfg = cfg
        n_enc = len(FAMILIES) if cfg["per_family_encoders"] else 1
        self.encoders = nn.ModuleDict({str(i): RepEncoder(n_channels, n_drills, cfg["drill_dim"],
                                                          cfg["hidden"], cfg["dropout"])
                                       for i in range(n_enc)})
        self.pool = Pool(cfg["pool"], cfg["hidden"])
        head_in = cfg["hidden"] + (n_attrs if cfg["use_attrs"] else 0)
        self.head = nn.Sequential(nn.Linear(head_in, 32), nn.ReLU(), nn.Dropout(cfg["dropout"]),
                                  nn.Linear(32, n_out))
    def encode_reps(self, x, drill, fam, mask):
        B, R, T, C = x.shape
        fx, fd = x.reshape(B * R, T, C), drill.reshape(-1)
        ff = fam.reshape(-1) if self.cfg["per_family_encoders"] else torch.zeros_like(fd)
        valid = mask.reshape(-1)
        out = x.new_zeros(B * R, self.cfg["hidden"])
        for key, enc in self.encoders.items():
            sel = (ff == int(key)) & valid
            if sel.any():
                out[sel] = enc(fx[sel], fd[sel])
        return out.reshape(B, R, -1)
    def forward(self, b):
        h = self.encode_reps(b["x"], b["drill"], b["fam"], b["mask"])
        v = self.pool(h, b["mask"])                      # player vector
        if self.cfg["use_attrs"]:
            v = torch.cat([v, b["attrs"]], dim=-1)
        return self.head(v)                              # (B, n_out)

## 10. Training

- **Loss**: masked, weighted MSE. Entries with `ym = 0` (family not run / too few routes) contribute nothing; `w` upweights players with more routes (their averages are less noisy).
- **Early stopping** uses a validation set carved out of the *training* players. Never early-stop on the test fold.
- **Regularization**: dropout, AdamW weight decay, small hidden size, gradient clipping. Consider data augmentation (small random time shifts or noise on `x`) as a variation.
- Targets are in separation units; standardizing `Y` per family using train statistics usually helps optimization (TODO).

In [ ]:
def masked_mse(pred, y, ym, w):
    num = (((pred - y) ** 2) * ym * w).sum()
    den = (ym * w).sum().clamp(min=1e-8)
    return num / den

def to_dev(b):
    return {k: v.to(DEVICE) for k, v in b.items()}

@torch.no_grad()
def predict(model, dl):
    model.eval()
    return torch.cat([model(to_dev(b)).cpu() for b in dl]).numpy()

@torch.no_grad()
def eval_loss(model, dl):
    model.eval()
    tot, n = 0.0, 0
    for b in dl:
        b = to_dev(b)
        tot += masked_mse(model(b), b["y"], b["ym"], b["w"]).item() * len(b["y"])
        n += len(b["y"])
    return tot / max(n, 1)

def fit(model, train_dl, val_dl, cfg):
    """[AI-SCAFFOLD] AdamW + early stopping on the inner validation loss; restores the best weights."""
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    best, best_state, bad = float("inf"), None, 0
    for ep in range(cfg["epochs"]):
        model.train()
        for b in train_dl:
            b = to_dev(b)
            opt.zero_grad()
            loss = masked_mse(model(b), b["y"], b["ym"], b["w"])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        v = eval_loss(model, val_dl)
        if v < best - 1e-5:
            best, bad = v, 0
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg["patience"]:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return best

## 11. Grouped cross-validation for the neural model

For every outer fold:
1. Split the training players into **inner train / inner validation** (about 85/15) for early stopping.
2. Fit scalers on inner-train players only.
3. Train `n_seeds` models with different seeds and **average their predictions** (small-data networks are high variance; report the spread across seeds too).
4. Store out-of-fold predictions for the test players.

Then score the OOF matrix with the same `score()` used for the baselines. Leakage checklist: scalers, drill/route mappings that depend on data, imputation, and early stopping must never see test-fold players.

In [ ]:
def run_cv(cfg, attrs, mode="kfold"):
    """[AI-SCAFFOLD] Returns an OOF prediction matrix (n_players, n_route_families)."""
    n = len(player_ids)
    oof = np.full_like(Y, np.nan)
    n_channels = reps[0]["x"].shape[1]
    for fold, (tr, te) in enumerate(get_splits(n, mode)):
        rng = np.random.RandomState(SEED + fold)
        perm = rng.permutation(tr)
        n_val = max(1, int(0.15 * len(perm)))
        inner_val, inner_tr = perm[:n_val], perm[n_val:]

        scalers = fit_scalers([player_ids[i] for i in inner_tr], reps_by_player, attrs[inner_tr])
        mk = lambda idx: DataLoader(
            PlayerDataset(idx, player_ids, reps_by_player, attrs, Y, YM, W, scalers),
            batch_size=cfg["batch_size"], shuffle=False, collate_fn=collate)
        dl_tr = DataLoader(PlayerDataset(inner_tr, player_ids, reps_by_player, attrs, Y, YM, W, scalers),
                           batch_size=cfg["batch_size"], shuffle=True, collate_fn=collate)
        dl_val, dl_te = mk(inner_val), mk(te)

        preds = []
        for s in range(cfg["n_seeds"]):
            set_seed(SEED + 100 * fold + s)
            model = PlayerModel(n_channels, len(DRILL_IDS), attrs.shape[1], Y.shape[1], cfg).to(DEVICE)
            fit(model, dl_tr, dl_val, cfg)
            preds.append(predict(model, dl_te))
        oof[te] = np.mean(preds, axis=0)
    return oof

# TODO: build `attrs` (n_players, n_attrs) = the same official numbers used in feature set A.
# attrs = X_A
# oof_C = run_cv(CFG, attrs)
# resC = score(oof_C, Y, YM); print(resC)
# TODO: put resA, resB, resC side by side (one table, with bootstrap CIs over players).

## 12. Variations and ablations (this is your experiment section)

Run each as a config change, using the same folds and seeds. Keep a results DataFrame (one row per experiment) and save it to disk after every run so a crashed Colab session does not cost you results.

| Variation | What it answers |
|---|---|
| `pool`: mean vs max vs weighted vs attention | Does attention over reps help, or is mean pooling enough? |
| `per_family_encoders`: False vs True | Shared LSTM vs one per drill family (bias-variance tradeoff) |
| drill embedding on/off | Does the model need to know which drill it is reading? |
| leave out one drill family at a time | Which families carry the signal? |
| `use_attrs`: with / without official numbers (and with draft pick) | Does the learned part add information beyond A? |
| `hidden`, `dropout`, `weight_decay` | Where does overfitting start? |
| rep selection: all reps vs best rep per drill | Does the model benefit from using all reps? |
| matched vs mismatched drill-route pairs | Football-facing hypothesis: does the WR route drill predict break-route separation better than the 40? |
| split mode: kfold vs draft_class | Does performance survive a realistic "next class" test? |
| LSTM vs a simple non-sequential model on summary features | Does the sequence model earn its complexity? |

Always include the baselines A and B in the same table. A null result (the neural model does not beat A or B) is a legitimate finding: report it honestly and discuss why (sample size, noise in targets, selection).

In [ ]:
EXPERIMENTS = {
    "pool_mean":       dict(pool="mean"),
    "pool_max":        dict(pool="max"),
    "pool_weighted":   dict(pool="weighted"),
    "pool_attention":  dict(pool="attention"),
    "per_family_enc":  dict(per_family_encoders=True),
    "no_attrs":        dict(use_attrs=False),
    # TODO: add drill-ablation entries (requires a way to filter reps by family before building the table)
}

all_results = []
def run_experiments(attrs, mode="kfold"):
    for name, over in EXPERIMENTS.items():
        cfg = {**CFG, **over}
        oof = run_cv(cfg, attrs, mode)
        r = score(oof, Y, YM)
        r.insert(0, "experiment", name)
        all_results.append(r)
        pd.concat(all_results).to_csv("results_so_far.csv", index=False)   # checkpoint after every run
    return pd.concat(all_results)

# results_df = run_experiments(attrs)
# summary = results_df.groupby("experiment")[["spearman", "r2"]].mean()

## 13. Interpretation (what makes this useful to a coach, not just a number)

- **Pooling weights.** For `weighted` / `attention` models, pull `model.pool.last_weights` on test players and aggregate by drill: which drills does the model lean on for each route family? Treat this as a hint, not proof. Confirm with the drill-ablation experiment.
- **Permutation importance.** Shuffle one drill family's reps across players in the test fold and measure the drop in Spearman.
- **Residual analysis.** Which WRs does the model under- and over-predict? Is it related to draft slot, route volume, or opt-outs?
- **Compare against A.** Show a scatter of A-only predictions vs C predictions for the same players. Where do they disagree, and who is right?
- **Figures to keep (they feed both your paper and the Big Data Bowl writeup):** (1) per-family Spearman for A / B / C with CIs, (2) pooling variants, (3) drill-ablation drops, (4) example rep traces with the attention weights, (5) one "scouting card" style summary.

In [ ]:
# TODO: after training a final model on all training players, run the test loader and collect pool weights.
# Example sketch:
#   model.eval(); _ = model(to_dev(batch)); w = model.pool.last_weights   # (B, R) or (B, R, R)
#   Map each rep back to its drill_id, group by drill name, and average the weight per (route family, drill).

## 14. Wrap-up checklist

**For DS340 (final paper, talk, code)**
- [ ] Say exactly which parts are lecture material (LSTMs, backprop, regularization, trees/ensembles for baselines) and which you taught yourselves (event alignment of tracking data, attention-based pooling over variable-size sets, grouped evaluation, masked losses).
- [ ] AI disclosure: matches what the code comments say (`# [AI-SCAFFOLD]` tags kept only where code is unchanged).
- [ ] The honest limits section: ~100 WRs, noisy targets, selection into the Combine, three draft classes.

**For the NFL Big Data Bowl (deadline Jan 6, 2027; check the Rules tab for the current requirements)**
- [ ] The writeup must use the Combine tracking data and explicitly link it to regular-season performance.
- [ ] Reframe results for football readers: lead with the question and the finding, show the scouting-style figure, keep the model description short.
- [ ] Publish a clean public notebook that runs top to bottom.